# DL vertex input image generation

This notebook is designed to take CSV files generated by the <code>PrepareTrainingSample</code> function of the <code>DlVertexingAlgorithm</code>. This algorithm generates CSV files for each of the U. V and W views and the code below will run over each of those files.
    
Most of the cells below will not need any editing, but at the very bottom of the notebook you will find some additional markdown that describes what you may need to edit (essentially just some file locations).

In [11]:
# Automatically reload external libraries that change
%reload_ext autoreload
%autoreload 2

# If a matplotlib plot command is issued, display the results in the notebook
%matplotlib inline

In [12]:
import cv2
import csv
import numpy as np
import os
import glob
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import matplotlib as mpl

In [13]:
wire_pitch = { "U": 0.46669998765, "V": 0.46669998765, "W": 0.479000002146 }
wire_pitch = { "UV": 0.3, "W": 0.3, "U": 0.3, "V": 0.3 }
drift_step = 0.5

# thresholds must be defined/imported wherever make_input_histogram is used;
# left as a module-level global to match the original code's expectations.
thresholds = None

def make_input_histogram(x, z, adc, vertex, x_bounds, z_bounds, image_size, view):
    global thresholds
    image_height, image_width = image_size
    x_min, x_max = x_bounds
    z_min, z_max = z_bounds
    
    # Update the span if image is too small
    r_span = np.sqrt((x_max - x_min)**2 + (z_max - z_min)**2)
    
    x_bins = np.linspace(x_min - 0.5 * drift_step, x_max + 0.5 * drift_step, image_width + 1)
    z_bins = np.linspace(z_min - 0.5 * wire_pitch[view], z_max + 0.5 * wire_pitch[view], image_height + 1)
    
    phx = np.digitize(x, x_bins) - 1
    phz = np.digitize(z, z_bins) - 1
        
    pvx = np.digitize(vertex[0], x_bins) - 1
    pvz = np.digitize(vertex[1], z_bins) - 1

    input_histogram, _, _ = np.histogram2d(z, x, bins=[z_bins, x_bins], weights=adc)
    input_histogram = input_histogram.astype(float)

    # ATTN: Need to set the dtype here or truth will be of type 8-bit uint before scaling, which will result in
    # incorrect distances for dr > 255
    truth_histogram = np.zeros_like(input_histogram, dtype=float)
    
    # Handle case where vertex is outside of the hit bounding box. Note, it may still be possible for the correction
    # to move the vertex outside the bounding box in the opposite direction (unlikely, but think about it)
    # other way to do this is to bound the image including the vertex location
    if pvx < 0: # underflow
        pvx = -(np.digitize(x_bins[0] + (x_bins[0] - vertex[0]), x_bins) - 1)
    elif pvx >= (len(x_bins) - 1): # overflow
        pvx = np.digitize(x_bins[-1] - (vertex[0] - x_bins[-1]), x_bins) - 1
    if pvz < 0: # underflow
        pvz = -(np.digitize(z_bins[0] + (z_bins[0] - vertex[1]), z_bins) - 1)
    elif pvz >= (len(z_bins) - 1): # overflow
        pvz = np.digitize(z_bins[-1] - (vertex[1] - z_bins[-1]), z_bins) - 1

    dr = np.sqrt((phx - pvx)**2 + (phz - pvz)**2)
    class_histogram = np.zeros_like(truth_histogram)
    for i in range(len(phx)):
        truth_histogram[phz[i], phx[i]] = dr[i]
    truth_min, truth_max = np.min(truth_histogram), np.max(truth_histogram)
    if truth_max > truth_min:
        truth_histogram = (truth_histogram - truth_min) / np.ceil(np.sqrt(2*(image_height - 1)**2))
        for i in range(len(phx)):
            cls = np.digitize(truth_histogram[phz[i], phx[i]], thresholds)
            class_histogram[phz[i], phx[i]] = cls if cls < len(thresholds) else len(thresholds) - 1
    else:
        class_histogram = np.zeros_like(input_histogram)

    return input_histogram, class_histogram.astype('uint8')

Task was destroyed but it is pending!
task: <Task pending name='Task-189' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at /opt/conda/lib/python3.10/site-packages/ipykernel/utils.py:76> wait_for=<Task pending name='Task-190' coro=<_async_in_context.<locals>.preserve_context() running at /opt/conda/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/conda/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/conda/lib/python3.10/tokenize.py:527: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  pseudomatch = _compile(PseudoToken).match(line, pos)
Task was destroyed but it is pending!
task: <Task pending name='Task-190' coro=<_async_in_context.<locals>.preserve_context() running at /opt/conda/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]>


In [14]:
def display(hits_x, hits_z, vrt_x, vrt_z):
    """Displays an image of the hits and vertices.
    
        Args:
            hits_x: list of x coordinates for hits
            hits_z: list of z coordinates for hits
            vrt_x: list of x coordinates for vertices
            vrt_z: list of z coordinates for vertices
    """
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(5,5))
    plt.scatter(hits_x, hits_z, c='black', s=20, alpha=1.0, label="Hits")
    plt.scatter(vrt_x, vrt_z, c='red', s=50, alpha=1.0, label="Vertices")
    plt.legend()
    plt.show()
    plt.close(fig)

def process_event(data, image_size, view):
    """Generate the training/validation set images for a single event.
    
        The input data has the format:
        N Vertices,M Hits,N*{vertex x, vertex y},M*{hit x, hit z, adc}
        
        The first vertex in the list is always the primary vertex
        
        Images are output to <output_folder>/Hits and <output_folder>/Truth
    
        Args:
            data: The event's fields (already trimmed of leading/trailing sentinel columns)
            image_size: (height, width) of the output histograms
            view: Detector view
    """
    nv_coords = 2
    nh_coords = 3
    nuance = int(data.pop(0))
    n_vertices = int(data.pop(0))
    
    v_start, v_finish = 0, nv_coords * n_vertices
    vx = np.array(data[v_start:v_finish:2], dtype=float)
    vz = np.array(data[v_start + 1:v_finish:2], dtype=float)
    b_start = v_finish
    x_min, x_max = float(data[b_start]), float(data[b_start + 1])
    z_min, z_max = float(data[b_start + 2]), float(data[b_start + 3])
    
    n_hits = int(data[b_start + 4])
    h_start, h_finish = b_start + 5, b_start + 5 + nh_coords * n_hits
    length = len(data[h_start:])
    
    if length != (n_hits * nh_coords):
        print('Missing information in input file')
        print(n_hits, length)
        return
    
    hx = np.array(data[h_start:h_finish:nh_coords], dtype=float)
    hz = np.array(data[h_start + 1:h_finish:nh_coords], dtype=float)
    hadc = np.array(data[h_start + 2:h_finish:nh_coords], dtype=float)

    if hx.size == 0 or hz.size == 0 or hadc.size == 0:
        return None
    
    return make_input_histogram(
        hx, hz, hadc, (vx[0], vz[0]),
        (x_min, x_max), (z_min, z_max), image_size, view
    )

def process_file(input_file, output_folder, view, image_size=(128, 128), shard_size=2000):
    
    """Generate the sharded training/validation set images for events.
    
        The input CSV file has the format:
        Date/Time,N Vertices,M Hits,N*{vertex x, vertex y},M*{hit x, hit y},EOL
        
        The first vertex in the list is always the primary vertex

        Writes to <output_folder>/Hits/shard_{n}.npy and <output_folder>/Truth/shard_{n}.npy,
        each holding a stacked array of up to `shard_size` samples, instead of one file
        per event. A single pass over the file is used both to count rows (for the tqdm
        total) and to process them.
    
        Args:
            input_file: a CSV file containing event information
            output_folder: The top-level folder for output images
            view: Detector view
            image_size: The output image size as a tuple (height, width) (default: (128, 128))
            shard_size: Number of events to accumulate per output shard (default: 2000)
    """
    
    hits_dir = os.path.join(output_folder, "Hits")
    truth_dir = os.path.join(output_folder, "Truth")
    os.makedirs(hits_dir, exist_ok=True)
    os.makedirs(truth_dir, exist_ok=True)
 
    # Single pass: count lines for the progress bar via file size heuristic is unreliable for
    # CSV, so if an accurate total matters, do a cheap line count without materializing lines.
    with open(input_file, 'r') as f:
        num_events = sum(1 for _ in f)
 
    hits_buf, truth_buf = [], []
    shard_idx = 0
 
    def flush():
        nonlocal hits_buf, truth_buf, shard_idx
        if not hits_buf:
            return
        np.savez_compressed(os.path.join(hits_dir, f"shard_{shard_idx}.npz"), np.stack(hits_buf))
        np.savez_compressed(os.path.join(truth_dir, f"shard_{shard_idx}.npy"), np.stack(truth_buf))
        hits_buf, truth_buf = [], []
        shard_idx += 1
 
    with open(input_file, 'r') as f:
        reader = csv.reader(f)
        for row in tqdm(reader, desc=f"Processing view {view}", miniters=100, total=num_events):
            result = process_event(row[1:-1], image_size, view)
            if result is None:
                continue
            input_histogram, truth_histogram = result
            hits_buf.append(input_histogram)
            truth_buf.append(truth_histogram)
            if len(hits_buf) >= shard_size:
                flush()
 
    flush()  # final partial shard


# Edit below this point

The details that might change between different contexts are the input and output file locations, the class thresholds and potentially the number of passes.

The input files are specified by the <code>file_prefix</code> variable - the <code>PrepareTrainingSample</code> function automatically tags the files with their respective views, so you should omit the view and file type from the specification.

The output location is specified by <code>global_path</code>, within which <code>Hit</code> and <code>Truth</code> folders will be created to store the input and target output images for training.

The thresholds are specified by the <code>thresholds</code> variable, a global variable referenced by <code>make_input_histograms</code>. It is critical that the values here match those specified in the Pandora XML specification for the vertexing algorithm.

In general, you will want to implement a two pass approach to networking, as this will likely greatly enhance vertex resolution, but if you may only want one pass due to higher pass dependence on earlier passes for CSV generation, you can just alter the <code>vertex_pass</code> loop list to run over the selected pass. There is no explicit dependency between the passes in this notebook.

Once you're happy with these values, you can just run the entire notebook from top to bottom and, after some time, you'll have a set of input/truth images that can be used to train the networks.

In [15]:
import matplotlib.pyplot as plt

plt.rcParams["axes.ymargin"] = 0.1
plt.rcParams["legend.frameon"] = False
plt.rcParams["xaxis.labellocation"] = "right"
plt.rcParams["yaxis.labellocation"] = "top"

# 1. ax.ticklabel_format(style='scientific', scilimits=(0,0))
# plt.rcParams['axes.formatter.style'] = 'scientific'
plt.rcParams['axes.formatter.limits'] = (-2, 3)

# 2. ax.ticklabel_format(useMathText=True)
plt.rcParams['axes.formatter.use_mathtext'] = True

# 3. ax.minorticks_on()
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

plt.rcParams['xtick.major.size'] = 6
plt.rcParams['ytick.major.size'] = 6

plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

plt.rcParams['xtick.top'] = True
plt.rcParams['ytick.right'] = True

# 2. ax.tick_params(which='minor', length=3, direction='in', right=True, top=True)
plt.rcParams['xtick.minor.size'] = 3
plt.rcParams['ytick.minor.size'] = 3

# Note: Direction, top, and right settings automatically apply to minor ticks 
# when set globally, but you can explicitly ensure they mirror major ticks.
plt.rcParams['xtick.minor.top'] = True
plt.rcParams['ytick.minor.right'] = True

plt.rcParams['axes.xmargin'] = 0.0

thresholds = [0., 0.00275, 0.00825, 0.01925, 0.03575, 0.05775, 0.08525, \
              0.12375, 0.15125, 0.20625, 0.26125, 0.31625, 0.37125, 0.42625, \
              0.50875, 0.59125, 0.67375, 0.75625, 0.85, 1.0]

# Pass 1: creating images for the first pass

In [16]:
for vertex_pass in [1]:
    for beam in tqdm(['BNB', 'NuMI']):
        for view in tqdm(['W', 'UV']):
            for flavour in tqdm(['numu', 'nue']):
                file_prefix = (
                    f'/exp/icarus/data/users/msotgia/vertexStudies/forTraining/pass{vertex_pass}/'
                    f'{beam}/{flavour}/ICARUS_DLVertexTrain_Pass{vertex_pass}_CaloHitList{view}.csv'
                )
                image_size = (256, 256) if vertex_pass == 1 else (192, 192)
        
                global_path = os.path.join(f"/home/msotgia/vertexOnEaf/ICARUS_DlVertex_fastLoader/{beam}/{flavour}/Pass{vertex_pass}", f"Images{view}")
                process_file(file_prefix, global_path, view, image_size = image_size, shard_size=10000)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Processing view W:   0%|          | 0/152421 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter serve

Processing view UV:   0%|          | 0/445296 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

